# Testing Notebook — Credit Risk Model Serving

**Nama:** rzky0x  
**Model:** Credit Risk Prediction (DNN Classifier)  
**Platform:** Railway (TensorFlow Serving)

---

Notebook ini digunakan untuk menguji model serving yang telah di-deploy di cloud. Kita akan mengirim prediction request ke REST API endpoint dan memverifikasi hasilnya.

## 1. Setup

Import library dan konfigurasi endpoint URL.

In [ ]:
import json
import requests
import base64
import numpy as np
import tensorflow as tf

In [ ]:
# Configuration - Update this with your Railway deployment URL
# Format: https://<your-app>.up.railway.app
SERVING_URL = 'https://mlops-credit-risk-production.up.railway.app'
MODEL_NAME = 'credit-risk-model'

# API Endpoints
METADATA_URL = f'{SERVING_URL}/v1/models/{MODEL_NAME}/metadata'
PREDICT_URL = f'{SERVING_URL}/v1/models/{MODEL_NAME}:predict'

print(f'Metadata URL: {METADATA_URL}')
print(f'Predict URL: {PREDICT_URL}')

## 2. Cek Model Metadata

Mengecek apakah model serving sudah aktif dan mendapatkan informasi model.

In [ ]:
# Check model metadata
try:
    response = requests.get(METADATA_URL, timeout=10)
    if response.status_code == 200:
        metadata = response.json()
        print('Model serving is ACTIVE!')
        print(json.dumps(metadata, indent=2))
    else:
        print(f'Error: Status code {response.status_code}')
        print(response.text)
except requests.exceptions.ConnectionError:
    print('Connection error. Make sure the serving URL is correct.')

## 3. Membuat Data Test

Membuat beberapa contoh data untuk prediction request.

In [ ]:
def create_tf_example(data):
    """Create a serialized tf.Example from a dictionary.
    
    Args:
        data: Dictionary with feature names as keys.
    
    Returns:
        Base64 encoded serialized tf.Example.
    """
    feature_dict = {}
    
    for key, value in data.items():
        if isinstance(value, str):
            feature_dict[key] = tf.train.Feature(
                bytes_list=tf.train.BytesList(value=[value.encode('utf-8')])
            )
        elif isinstance(value, float):
            feature_dict[key] = tf.train.Feature(
                float_list=tf.train.FloatList(value=[value])
            )
        elif isinstance(value, int):
            feature_dict[key] = tf.train.Feature(
                int64_list=tf.train.Int64List(value=[value])
            )
    
    example = tf.train.Example(
        features=tf.train.Features(feature=feature_dict)
    )
    
    serialized = example.SerializeToString()
    return base64.b64encode(serialized).decode('utf-8')

print('Helper function created.')

In [ ]:
# Test data - Low risk borrower
low_risk_data = {
    'person_age': 35,
    'person_income': 80000,
    'person_home_ownership': 'MORTGAGE',
    'person_emp_length': 10.0,
    'loan_intent': 'EDUCATION',
    'loan_grade': 'A',
    'loan_amnt': 5000,
    'loan_int_rate': 7.5,
    'loan_percent_income': 0.06,
    'cb_person_default_on_file': 'N',
    'cb_person_cred_hist_length': 15,
}

# Test data - High risk borrower
high_risk_data = {
    'person_age': 22,
    'person_income': 15000,
    'person_home_ownership': 'RENT',
    'person_emp_length': 1.0,
    'loan_intent': 'PERSONAL',
    'loan_grade': 'D',
    'loan_amnt': 10000,
    'loan_int_rate': 15.0,
    'loan_percent_income': 0.67,
    'cb_person_default_on_file': 'Y',
    'cb_person_cred_hist_length': 2,
}

# Encode examples
low_risk_example = create_tf_example(low_risk_data)
high_risk_example = create_tf_example(high_risk_data)

print('Test examples created successfully.')
print(f'Low risk example (base64 length): {len(low_risk_example)}')
print(f'High risk example (base64 length): {len(high_risk_example)}')

## 4. Mengirim Prediction Request

Mengirim data ke model serving dan mendapatkan prediksi risiko kredit.

In [ ]:
# Send prediction request
def predict(serving_url, model_name, examples):
    """Send prediction request to TF Serving.
    
    Args:
        serving_url: Base URL of the serving endpoint.
        model_name: Name of the model.
        examples: List of base64-encoded tf.Example strings.
    
    Returns:
        Prediction results as JSON.
    """
    predict_url = f'{serving_url}/v1/models/{model_name}:predict'
    
    payload = {
        'signature_name': 'serving_default',
        'instances': [{'examples': {'b64': ex}} for ex in examples]
    }
    
    headers = {'Content-Type': 'application/json'}
    
    response = requests.post(
        predict_url,
        json=payload,
        headers=headers,
        timeout=30
    )
    
    return response.json()

print('Prediction function created.')

In [ ]:
# Make predictions
try:
    results = predict(
        SERVING_URL,
        MODEL_NAME,
        [low_risk_example, high_risk_example]
    )
    
    print('=== Prediction Results ===')
    print(json.dumps(results, indent=2))
    
    if 'predictions' in results:
        predictions = results['predictions']
        labels = ['Non-Default (Low Risk)', 'Default (High Risk)']
        
        for i, (pred, label) in enumerate(zip(predictions, labels)):
            prob = pred['outputs'][0] if isinstance(pred['outputs'], list) else pred['outputs']
            risk_label = 'DEFAULT' if prob > 0.5 else 'NON-DEFAULT'
            print(f'\nTest Case {i+1} ({label}):')
            print(f'  Prediction probability: {prob:.4f}')
            print(f'  Predicted class: {risk_label}')
    else:
        print('Unexpected response format.')
        
except requests.exceptions.ConnectionError:
    print('Connection error. Make sure the serving URL is correct and the service is running.')
except Exception as e:
    print(f'Error: {e}')

## 5. Cek Monitoring Endpoint

Memverifikasi bahwa Prometheus metrics endpoint aktif.

In [ ]:
# Check monitoring endpoint
MONITORING_URL = f'{SERVING_URL}/monitoring/prometheus/metrics'

try:
    response = requests.get(MONITORING_URL, timeout=10)
    if response.status_code == 200:
        metrics_text = response.text
        # Show first 500 characters of metrics
        print('Prometheus metrics endpoint is ACTIVE!')
        print(f'Metrics (first 500 chars):\n{metrics_text[:500]}...')
    else:
        print(f'Error: Status code {response.status_code}')
except requests.exceptions.ConnectionError:
    print('Connection error. Monitoring endpoint may not be configured.')

## 6. Kesimpulan

Model serving berhasil diuji dengan prediction request. Berikut ringkasannya:

| Test | Deskripsi | Expected | Result |
|------|-----------|----------|--------|
| Metadata | Cek status model | Active | ✅ |
| Low Risk | Peminjam dengan profil baik | Non-Default | ✅ |
| High Risk | Peminjam dengan profil berisiko | Default | ✅ |
| Monitoring | Prometheus metrics | Active | ✅ |